In [1]:
from __future__ import annotations

import os
import random
import copy
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# ============================================================
# 0. Configuration
# ============================================================
SEED = 42

DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/datasets/UCI-HAR"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Colab Notebooks/HAR/BG-HAR/pt_files/uci_har.pt"

BATCH_SIZE = 128
EPOCHS = 80
GATE_WARMUP_EPOCHS = 10
LR = 2e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 5.0

HIDDEN_DIM = 64
TOTAL_MAMBA_BLOCKS = 3
EARLY_EXIT_BLOCKS = 1

DROPOUT = 0.05

EARLY_LOSS_WEIGHT = 0.30
GATE_LOSS_WEIGHT = 0.45
COMPUTE_PENALTY_WEIGHT = 0.01

BENEFIT_MARGIN = 0.07
MAX_FULL_ROUTE_RATIO = 0.45
GATE_HIDDEN_DIM = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)



# ============================================================
# 1. Reproducibility
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)


# ============================================================
# 2. Use existing UCI-HAR dataset directory only
# ============================================================

def is_uci_har_root(path: str) -> bool:
    train_inertial = os.path.join(path, "train", "Inertial Signals")
    test_inertial = os.path.join(path, "test", "Inertial Signals")

    return (
        os.path.isdir(train_inertial)
        and os.path.isdir(test_inertial)
    )

# ============================================================
# 3. Load UCI-HAR raw inertial signals
# ============================================================

SIGNAL_FILES = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]


def load_txt_matrix(path: str):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required file not found:\n{path}")
    return pd.read_csv(path, sep=r"\s+", header=None).values.astype(np.float32)


def load_split(root: str, split: str):
    signal_dir = os.path.join(root, split, "Inertial Signals")

    signals = []
    for name in SIGNAL_FILES:
        file_path = os.path.join(signal_dir, f"{name}_{split}.txt")
        arr = load_txt_matrix(file_path)
        signals.append(arr)

    X = np.stack(signals, axis=1)  # (N, C, T)

    y_path = os.path.join(root, split, f"y_{split}.txt")
    y = load_txt_matrix(y_path).astype(np.int64).reshape(-1) - 1

    subject_path = os.path.join(root, split, f"subject_{split}.txt")
    subject = load_txt_matrix(subject_path).astype(np.int64).reshape(-1)

    return X, y, subject

X_train_full, y_train_full, subject_train_full = load_split(DATA_ROOT, "train")
X_test, y_test, subject_test = load_split(DATA_ROOT, "test")
ACTIVITY_NAMES = ["WALK", "UP", "DOWN", "SIT", "STAND", "LAY"]

NUM_CLASSES = len(ACTIVITY_NAMES)
NUM_CHANNELS = X_train_full.shape[1]
SEQ_LEN = X_train_full.shape[2]

print("Full train X:", X_train_full.shape)
print("Test X      :", X_test.shape)
print("Activities  :", ACTIVITY_NAMES)


# ============================================================
# 4. Subject-disjoint split inside official training set
# ============================================================

def subject_disjoint_train_val_split(X, y, subject, val_ratio=0.20, seed=11):
    unique_subjects = np.array(sorted(np.unique(subject)))
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_subjects)

    n_val = max(1, int(round(len(unique_subjects) * val_ratio)))
    val_subjects = set(unique_subjects[:n_val].tolist())

    val_mask = np.array([s in val_subjects for s in subject])
    train_mask = ~val_mask

    return (
        X[train_mask], y[train_mask], subject[train_mask],
        X[val_mask], y[val_mask], subject[val_mask],
        sorted(list(val_subjects)),
    )


X_train, y_train, subject_train, X_val, y_val, subject_val, val_subjects = subject_disjoint_train_val_split(
    X_train_full,
    y_train_full,
    subject_train_full,
    val_ratio=0.20,
    seed=SEED,
)

print("Train subjects for fitting:", sorted(np.unique(subject_train).tolist()))
print("Val subjects for gate calibration:", val_subjects)
print("Train X:", X_train.shape)
print("Val X  :", X_val.shape)


# ============================================================
# 5. Train-only normalization
# ============================================================

train_mean = X_train.mean(axis=(0, 2), keepdims=True)
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

print("Normalized train mean:", float(X_train.mean()))
print("Normalized train std :", float(X_train.std()))


# ============================================================
# 6. Dataset and dataloader
# ============================================================

class HARWindowDataset(Dataset):
    def __init__(self, X, y, subject=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

        if subject is None:
            self.subject = torch.zeros(len(y), dtype=torch.long)
        else:
            self.subject = torch.tensor(subject, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.subject[idx]


train_dataset = HARWindowDataset(X_train, y_train, subject_train)
val_dataset = HARWindowDataset(X_val, y_val, subject_val)
test_dataset = HARWindowDataset(X_test, y_test, subject_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)


# ============================================================
# 7. Mamba-style block
# ============================================================

class MiniMambaBlock(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float = 0.2, conv_kernel: int = 5):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.conv_kernel = conv_kernel

        self.norm = nn.LayerNorm(hidden_dim)
        self.in_proj = nn.Linear(hidden_dim, hidden_dim * 2)

        self.depthwise_conv = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=conv_kernel,
            padding=conv_kernel // 2,
            groups=hidden_dim,
            bias=True,
        )

        self.dt_proj = nn.Linear(hidden_dim, hidden_dim)
        self.b_proj = nn.Linear(hidden_dim, hidden_dim)
        self.c_proj = nn.Linear(hidden_dim, hidden_dim)

        self.A_log = nn.Parameter(torch.zeros(hidden_dim))
        self.D = nn.Parameter(torch.ones(hidden_dim))

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def selective_scan(self, u):
        B, T, H = u.shape

        dt = F.softplus(self.dt_proj(u)) + 1e-4
        b_t = torch.tanh(self.b_proj(u))
        c_t = torch.tanh(self.c_proj(u))

        A = torch.exp(self.A_log).view(1, H)

        state = torch.zeros(B, H, device=u.device, dtype=u.dtype)
        outputs = []

        for t in range(T):
            dt_cur = dt[:, t, :]
            u_cur = u[:, t, :]
            b_cur = b_t[:, t, :]
            c_cur = c_t[:, t, :]

            a_bar = torch.exp(-dt_cur * A)
            b_bar = (1.0 - a_bar) * b_cur

            state = a_bar * state + b_bar * u_cur
            y_cur = c_cur * state + self.D.view(1, H) * u_cur

            outputs.append(y_cur)

        y = torch.stack(outputs, dim=1)
        return y

    def forward(self, x):
        residual = x

        x = self.norm(x)

        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)

        u = self.depthwise_conv(u.transpose(1, 2)).transpose(1, 2)
        u = F.silu(u)

        y = self.selective_scan(u)
        y = y * F.silu(z)

        y = self.out_proj(y)
        y = self.dropout(y)

        return residual + y


# ============================================================
# 8. Benefit-gated shared Mamba model
# ============================================================

class BenefitGatedSharedMambaHAR(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_dim: int = 64,
        total_blocks: int = 3,
        early_exit_blocks: int = 1,
        dropout: float = 0.2,
        gate_hidden_dim: int = 32,
    ):
        super().__init__()

        assert 1 <= early_exit_blocks < total_blocks

        self.in_channels = in_channels
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.total_blocks = total_blocks
        self.early_exit_blocks = early_exit_blocks

        self.input_proj = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )

        self.blocks = nn.ModuleList([
            MiniMambaBlock(hidden_dim=hidden_dim, dropout=dropout, conv_kernel=5)
            for _ in range(total_blocks)
        ])

        self.readout_norm = nn.LayerNorm(hidden_dim)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

        gate_input_dim = hidden_dim + num_classes + 6

        self.benefit_gate = nn.Sequential(
            nn.LayerNorm(gate_input_dim),
            nn.Linear(gate_input_dim, gate_hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(gate_hidden_dim, 1),
        )

    def pool_feature(self, z):
        z = self.readout_norm(z)
        h = z.mean(dim=1)
        return h

    def classify_from_feature(self, h):
        return self.classifier(h)

    def build_gate_input(self, x, h_early, early_logits):
        prob = F.softmax(early_logits, dim=1)
        top2 = torch.topk(prob, k=2, dim=1).values

        confidence = top2[:, 0:1]
        margin = top2[:, 0:1] - top2[:, 1:2]
        entropy = -(prob * torch.log(prob + 1e-8)).sum(dim=1, keepdim=True)
        entropy = entropy / math.log(self.num_classes)

        temporal_energy = (x[:, :, 1:] - x[:, :, :-1]).pow(2).mean(dim=(1, 2)).unsqueeze(1)
        signal_energy = x.pow(2).mean(dim=(1, 2)).unsqueeze(1)
        abs_mean = x.abs().mean(dim=(1, 2)).unsqueeze(1)

        gate_input = torch.cat(
            [
                h_early,
                early_logits,
                confidence,
                margin,
                entropy,
                temporal_energy,
                signal_energy,
                abs_mean,
            ],
            dim=1,
        )

        return gate_input

    def forward_all(self, x):
        z = self.input_proj(x)
        z = z.transpose(1, 2)

        early_logits = None
        h_early = None

        for i, block in enumerate(self.blocks):
            z = block(z)

            if i + 1 == self.early_exit_blocks:
                h_early = self.pool_feature(z)
                early_logits = self.classify_from_feature(h_early)

        gate_input = self.build_gate_input(x, h_early, early_logits)
        gate_logit = self.benefit_gate(gate_input).squeeze(1)
        gate_prob = torch.sigmoid(gate_logit)

        h_final = self.pool_feature(z)
        final_logits = self.classify_from_feature(h_final)

        return early_logits, final_logits, gate_logit, gate_prob

    @torch.no_grad()
    def forward_dynamic(self, x, benefit_tau: float):
        z = self.input_proj(x)
        z = z.transpose(1, 2)

        for i in range(self.early_exit_blocks):
            z = self.blocks[i](z)

        h_early = self.pool_feature(z)
        early_logits = self.classify_from_feature(h_early)

        gate_input = self.build_gate_input(x, h_early, early_logits)
        gate_logit = self.benefit_gate(gate_input).squeeze(1)
        gate_prob = torch.sigmoid(gate_logit)

        full_mask = gate_prob >= benefit_tau

        output_logits = early_logits.clone()
        route = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        if full_mask.any():
            z_full = z[full_mask]

            for i in range(self.early_exit_blocks, self.total_blocks):
                z_full = self.blocks[i](z_full)

            h_full = self.pool_feature(z_full)
            full_logits = self.classify_from_feature(h_full)

            output_logits[full_mask] = full_logits
            route[full_mask] = 1

        return output_logits, route, gate_prob


model = BenefitGatedSharedMambaHAR(
    in_channels=NUM_CHANNELS,
    num_classes=NUM_CLASSES,
    hidden_dim=HIDDEN_DIM,
    total_blocks=TOTAL_MAMBA_BLOCKS,
    early_exit_blocks=EARLY_EXIT_BLOCKS,
    dropout=DROPOUT,
    gate_hidden_dim=GATE_HIDDEN_DIM,
).to(DEVICE)

print(model)


# ============================================================
# 9. Parameter count
# ============================================================

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


print("Total trainable parameters:", count_params(model))
print("Input projection parameters:", count_params(model.input_proj))
print("Mamba block parameters:", count_params(model.blocks))
print("Shared classifier parameters:", count_params(model.classifier))
print("Benefit gate parameters:", count_params(model.benefit_gate))


# ============================================================
# 10. FLOPs estimation
# ============================================================

def flops_linear(in_dim, out_dim):
    return 2 * in_dim * out_dim


def flops_conv1d(in_ch, out_ch, kernel_size, out_len, groups=1):
    return 2 * out_len * out_ch * (in_ch // groups) * kernel_size


def estimate_minimamba_block_flops(T, H, conv_kernel=5):
    layernorm = 5 * T * H

    in_proj = T * flops_linear(H, 2 * H)
    depthwise_conv = flops_conv1d(H, H, conv_kernel, T, groups=H)

    dt_proj = T * flops_linear(H, H)
    b_proj = T * flops_linear(H, H)
    c_proj = T * flops_linear(H, H)

    selective_scan_elementwise = 35 * T * H

    out_proj = T * flops_linear(H, H)
    residual_add = T * H

    total = (
        layernorm
        + in_proj
        + depthwise_conv
        + dt_proj
        + b_proj
        + c_proj
        + selective_scan_elementwise
        + out_proj
        + residual_add
    )

    return total


def estimate_benefit_gate_flops(H, K, gate_hidden_dim):
    gate_input_dim = H + K + 6

    scalar_feature_ops = 40 * K + 20
    gate_norm = 5 * gate_input_dim
    gate_fc1 = flops_linear(gate_input_dim, gate_hidden_dim)
    gate_act = gate_hidden_dim
    gate_fc2 = flops_linear(gate_hidden_dim, 1)
    sigmoid = 5

    return scalar_feature_ops + gate_norm + gate_fc1 + gate_act + gate_fc2 + sigmoid


def estimate_route_flops(
    C,
    T,
    H,
    K,
    total_blocks,
    early_blocks,
    gate_hidden_dim,
    conv_kernel=5,
):
    input_proj = flops_conv1d(C, H, kernel_size=1, out_len=T, groups=1)
    block_flops = estimate_minimamba_block_flops(T=T, H=H, conv_kernel=conv_kernel)

    readout_norm = 5 * T * H
    gap = T * H
    classifier = flops_linear(H, K)

    early_head = readout_norm + gap + classifier
    full_head = readout_norm + gap + classifier

    benefit_gate = estimate_benefit_gate_flops(
        H=H,
        K=K,
        gate_hidden_dim=gate_hidden_dim,
    )

    early_route = (
        input_proj
        + early_blocks * block_flops
        + early_head
        + benefit_gate
    )

    full_route_dynamic = (
        input_proj
        + early_blocks * block_flops
        + early_head
        + benefit_gate
        + (total_blocks - early_blocks) * block_flops
        + full_head
    )

    full_model_once = (
        input_proj
        + total_blocks * block_flops
        + full_head
    )

    return {
        "input_projection": input_proj,
        "one_mamba_block": block_flops,
        "early_head": early_head,
        "benefit_gate": benefit_gate,
        "early_route": early_route,
        "full_route_dynamic": full_route_dynamic,
        "full_model_once": full_model_once,
    }


flops_info = estimate_route_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
    total_blocks=TOTAL_MAMBA_BLOCKS,
    early_blocks=EARLY_EXIT_BLOCKS,
    gate_hidden_dim=GATE_HIDDEN_DIM,
    conv_kernel=5,
)

print("\nApproximate per-sample FLOPs")
print("Input projection FLOPs :", f"{flops_info['input_projection'] / 1e6:.4f} M")
print("One Mamba block FLOPs  :", f"{flops_info['one_mamba_block'] / 1e6:.4f} M")
print("Benefit gate FLOPs     :", f"{flops_info['benefit_gate'] / 1e6:.4f} M")
print("Early route FLOPs      :", f"{flops_info['early_route'] / 1e6:.4f} M")
print("Full route FLOPs       :", f"{flops_info['full_route_dynamic'] / 1e6:.4f} M")
print("Full model once FLOPs  :", f"{flops_info['full_model_once'] / 1e6:.4f} M")


# ============================================================
# 11. Training utilities
# ============================================================
criterion_cls = nn.CrossEntropyLoss()
criterion_gate = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)


def make_benefit_target(early_logits, final_logits, y, benefit_margin=0.05):
    with torch.no_grad():
        early_ce = F.cross_entropy(early_logits, y, reduction="none")
        final_ce = F.cross_entropy(final_logits, y, reduction="none")

        early_pred = early_logits.argmax(dim=1)
        final_pred = final_logits.argmax(dim=1)

        corrected_by_full = (early_pred != y) & (final_pred == y)
        meaningful_loss_gain = (early_ce - final_ce) > benefit_margin
        full_harms_early = (early_pred == y) & (final_pred != y)

        target = corrected_by_full | (meaningful_loss_gain & (~full_harms_early))
        target = target.float()

    return target


def train_one_epoch(model, loader, epoch):
    model.train()

    total_loss = 0.0
    total_gate_loss = 0.0
    total_gate_target = 0.0
    total_gate_prob = 0.0

    all_y = []
    all_early_pred = []
    all_final_pred = []

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        early_logits, final_logits, gate_logit, gate_prob = model.forward_all(x)

        early_loss = criterion_cls(early_logits, y)
        final_loss = criterion_cls(final_logits, y)

        benefit_target = make_benefit_target(
            early_logits=early_logits,
            final_logits=final_logits,
            y=y,
            benefit_margin=BENEFIT_MARGIN,
        )

        gate_loss = criterion_gate(gate_logit, benefit_target)
        compute_penalty = gate_prob.mean()

        gate_weight = GATE_LOSS_WEIGHT * min(1.0, epoch / GATE_WARMUP_EPOCHS)

        loss = (
            final_loss
            + EARLY_LOSS_WEIGHT * early_loss
            + gate_weight * gate_loss
            + COMPUTE_PENALTY_WEIGHT * compute_penalty
        )

        loss.backward()

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_gate_loss += gate_loss.item() * x.size(0)
        total_gate_target += benefit_target.mean().item() * x.size(0)
        total_gate_prob += gate_prob.mean().item() * x.size(0)

        early_pred = early_logits.argmax(dim=1)
        final_pred = final_logits.argmax(dim=1)

        all_y.append(y.detach().cpu().numpy())
        all_early_pred.append(early_pred.detach().cpu().numpy())
        all_final_pred.append(final_pred.detach().cpu().numpy())

    all_y = np.concatenate(all_y)
    all_early_pred = np.concatenate(all_early_pred)
    all_final_pred = np.concatenate(all_final_pred)

    return {
        "loss": total_loss / len(loader.dataset),
        "gate_loss": total_gate_loss / len(loader.dataset),
        "gate_target_ratio": total_gate_target / len(loader.dataset),
        "gate_prob_mean": total_gate_prob / len(loader.dataset),
        "early_acc": accuracy_score(all_y, all_early_pred),
        "early_macro_f1": f1_score(all_y, all_early_pred, average="macro"),
        "final_acc": accuracy_score(all_y, all_final_pred),
        "final_macro_f1": f1_score(all_y, all_final_pred, average="macro"),
        "gate_weight": gate_weight,
    }


@torch.no_grad()
def evaluate_all(model, loader):
    model.eval()

    total_loss = 0.0
    total_gate_loss = 0.0

    all_y = []
    all_early_pred = []
    all_final_pred = []
    all_gate_prob = []
    all_gate_target = []
    all_subject = []

    for x, y, subject in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        early_logits, final_logits, gate_logit, gate_prob = model.forward_all(x)

        early_loss = criterion_cls(early_logits, y)
        final_loss = criterion_cls(final_logits, y)

        benefit_target = make_benefit_target(
            early_logits=early_logits,
            final_logits=final_logits,
            y=y,
            benefit_margin=BENEFIT_MARGIN,
        )

        gate_loss = criterion_gate(gate_logit, benefit_target)

        loss = (
            final_loss
            + EARLY_LOSS_WEIGHT * early_loss
            + GATE_LOSS_WEIGHT * gate_loss
            + COMPUTE_PENALTY_WEIGHT * gate_prob.mean()
        )

        total_loss += loss.item() * x.size(0)
        total_gate_loss += gate_loss.item() * x.size(0)

        early_pred = early_logits.argmax(dim=1)
        final_pred = final_logits.argmax(dim=1)

        all_y.append(y.detach().cpu().numpy())
        all_early_pred.append(early_pred.detach().cpu().numpy())
        all_final_pred.append(final_pred.detach().cpu().numpy())
        all_gate_prob.append(gate_prob.detach().cpu().numpy())
        all_gate_target.append(benefit_target.detach().cpu().numpy())
        all_subject.append(subject.numpy())

    all_y = np.concatenate(all_y)
    all_early_pred = np.concatenate(all_early_pred)
    all_final_pred = np.concatenate(all_final_pred)
    all_gate_prob = np.concatenate(all_gate_prob)
    all_gate_target = np.concatenate(all_gate_target)
    all_subject = np.concatenate(all_subject)

    return {
        "loss": total_loss / len(loader.dataset),
        "gate_loss": total_gate_loss / len(loader.dataset),
        "y_true": all_y,
        "early_pred": all_early_pred,
        "final_pred": all_final_pred,
        "gate_prob": all_gate_prob,
        "gate_target": all_gate_target,
        "subject": all_subject,
        "early_acc": accuracy_score(all_y, all_early_pred),
        "early_macro_f1": f1_score(all_y, all_early_pred, average="macro"),
        "final_acc": accuracy_score(all_y, all_final_pred),
        "final_macro_f1": f1_score(all_y, all_final_pred, average="macro"),
        "gate_target_ratio": float(all_gate_target.mean()),
        "gate_prob_mean": float(all_gate_prob.mean()),
    }


def dynamic_metrics_from_outputs(y_true, early_pred, final_pred, gate_prob, tau):
    full_mask = gate_prob >= tau

    dynamic_pred = early_pred.copy()
    dynamic_pred[full_mask] = final_pred[full_mask]

    acc = accuracy_score(y_true, dynamic_pred)
    mf1 = f1_score(y_true, dynamic_pred, average="macro")
    full_ratio = float(full_mask.mean())

    avg_flops = (
        (1.0 - full_ratio) * flops_info["early_route"]
        + full_ratio * flops_info["full_route_dynamic"]
    )

    return {
        "tau": float(tau),
        "acc": acc,
        "macro_f1": mf1,
        "full_route_ratio": full_ratio,
        "avg_flops": avg_flops,
        "dynamic_pred": dynamic_pred,
        "full_mask": full_mask,
    }


def calibrate_benefit_gate_threshold(
    y_true,
    early_pred,
    final_pred,
    gate_prob,
    max_full_route_ratio=0.45,
):
    candidate_taus = np.unique(
        np.concatenate([
            np.quantile(gate_prob, np.linspace(0.01, 0.99, 199)),
            np.array([gate_prob.min() - 1e-6, gate_prob.max() + 1e-6]),
        ])
    )

    records = []

    for tau in candidate_taus:
        m = dynamic_metrics_from_outputs(
            y_true=y_true,
            early_pred=early_pred,
            final_pred=final_pred,
            gate_prob=gate_prob,
            tau=tau,
        )
        records.append(m)

    feasible = [
        r for r in records
        if r["full_route_ratio"] <= max_full_route_ratio
    ]

    if len(feasible) > 0:
        best = sorted(
            feasible,
            key=lambda r: (r["macro_f1"], -r["avg_flops"]),
            reverse=True,
        )[0]
    else:
        best = sorted(
            records,
            key=lambda r: (r["macro_f1"], -r["avg_flops"]),
            reverse=True,
        )[0]

    return best, records


@torch.no_grad()
def evaluate_dynamic(model, loader, benefit_tau):
    model.eval()

    all_y = []
    all_pred = []
    all_route = []
    all_gate_prob = []
    all_subject = []

    for x, y, subject in loader:
        x = x.to(DEVICE, non_blocking=True)

        logits, route, gate_prob = model.forward_dynamic(
            x,
            benefit_tau=benefit_tau,
        )

        pred = logits.argmax(dim=1)

        all_y.append(y.numpy())
        all_pred.append(pred.detach().cpu().numpy())
        all_route.append(route.detach().cpu().numpy())
        all_gate_prob.append(gate_prob.detach().cpu().numpy())
        all_subject.append(subject.numpy())

    all_y = np.concatenate(all_y)
    all_pred = np.concatenate(all_pred)
    all_route = np.concatenate(all_route)
    all_gate_prob = np.concatenate(all_gate_prob)
    all_subject = np.concatenate(all_subject)

    full_ratio = float(all_route.mean())

    avg_flops = (
        (1.0 - full_ratio) * flops_info["early_route"]
        + full_ratio * flops_info["full_route_dynamic"]
    )

    return {
        "y_true": all_y,
        "y_pred": all_pred,
        "route": all_route,
        "gate_prob": all_gate_prob,
        "subject": all_subject,
        "acc": accuracy_score(all_y, all_pred),
        "macro_f1": f1_score(all_y, all_pred, average="macro"),
        "full_route_ratio": full_ratio,
        "early_route_ratio": 1.0 - full_ratio,
        "avg_flops": avg_flops,
    }


# ============================================================
# 12. Train
# ============================================================

best_val_final_mf1 = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    train_result = train_one_epoch(model, train_loader, epoch)
    val_result = evaluate_all(model, val_loader)

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_result["loss"],
        "train_gate_loss": train_result["gate_loss"],
        "train_gate_target_ratio": train_result["gate_target_ratio"],
        "train_gate_prob_mean": train_result["gate_prob_mean"],
        "train_early_acc": train_result["early_acc"],
        "train_early_macro_f1": train_result["early_macro_f1"],
        "train_final_acc": train_result["final_acc"],
        "train_final_macro_f1": train_result["final_macro_f1"],
        "val_early_acc": val_result["early_acc"],
        "val_early_macro_f1": val_result["early_macro_f1"],
        "val_final_acc": val_result["final_acc"],
        "val_final_macro_f1": val_result["final_macro_f1"],
        "val_gate_target_ratio": val_result["gate_target_ratio"],
        "val_gate_prob_mean": val_result["gate_prob_mean"],
        "lr": scheduler.get_last_lr()[0],
    }

    history.append(row)

    if val_result["final_macro_f1"] > best_val_final_mf1:
        best_val_final_mf1 = val_result["final_macro_f1"]
        best_state = copy.deepcopy(model.state_dict())

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(
            f"Epoch {epoch:03d} | "
            f"Train early F1 {train_result['early_macro_f1']:.4f} "
            f"final F1 {train_result['final_macro_f1']:.4f} | "
            f"Val early F1 {val_result['early_macro_f1']:.4f} "
            f"final F1 {val_result['final_macro_f1']:.4f} | "
            f"Gate target {val_result['gate_target_ratio']:.3f} "
            f"gate prob {val_result['gate_prob_mean']:.3f}"
        )

model.load_state_dict(best_state)

print("\nBest validation final Macro-F1:", best_val_final_mf1)

history_df = pd.DataFrame(history)

print("\nTraining history tail:")
try:
    display(history_df.tail(10))
except Exception:
    print(history_df.tail(10).to_string(index=False))


# ============================================================
# 13. Calibrate benefit-gate threshold on validation set
# ============================================================

val_outputs = evaluate_all(model, val_loader)

best_tau_record, tau_records = calibrate_benefit_gate_threshold(
    y_true=val_outputs["y_true"],
    early_pred=val_outputs["early_pred"],
    final_pred=val_outputs["final_pred"],
    gate_prob=val_outputs["gate_prob"],
    max_full_route_ratio=MAX_FULL_ROUTE_RATIO,
)

benefit_tau = best_tau_record["tau"]

print("\nCalibrated benefit-gate threshold")
print("Tau:", benefit_tau)
print("Val dynamic Acc:", best_tau_record["acc"])
print("Val dynamic Macro-F1:", best_tau_record["macro_f1"])
print("Val full-route ratio:", best_tau_record["full_route_ratio"])
print("Val average FLOPs:", f"{best_tau_record['avg_flops'] / 1e6:.4f} M")

tau_df = pd.DataFrame([
    {
        "tau": r["tau"],
        "macro_f1": r["macro_f1"],
        "acc": r["acc"],
        "full_route_ratio": r["full_route_ratio"],
        "avg_flops_M": r["avg_flops"] / 1e6,
    }
    for r in tau_records
])

tau_df = tau_df.sort_values(["macro_f1", "avg_flops_M"], ascending=[False, True])

print("\nTop threshold candidates:")
try:
    display(tau_df.head(10))
except Exception:
    print(tau_df.head(10).to_string(index=False))


# ============================================================
# 14. Final test evaluation
# ============================================================

test_all = evaluate_all(model, test_loader)
test_dynamic = evaluate_dynamic(model, test_loader, benefit_tau=benefit_tau)

print("\n================ Final Test Results ================")
print("Early-only Acc      :", test_all["early_acc"])
print("Early-only Macro-F1 :", test_all["early_macro_f1"])
print("Full-only Acc       :", test_all["final_acc"])
print("Full-only Macro-F1  :", test_all["final_macro_f1"])
print("Dynamic Acc         :", test_dynamic["acc"])
print("Dynamic Macro-F1    :", test_dynamic["macro_f1"])

print("\n================ Benefit Gate ================")
print("Test gate target ratio:", test_all["gate_target_ratio"])
print("Test gate prob mean   :", test_all["gate_prob_mean"])
print("Benefit threshold tau :", benefit_tau)

print("\n================ Route FLOPs ================")
print("Early route FLOPs per sample :", f"{flops_info['early_route'] / 1e6:.4f} M")
print("Full route FLOPs per sample  :", f"{flops_info['full_route_dynamic'] / 1e6:.4f} M")
print("Full model once FLOPs        :", f"{flops_info['full_model_once'] / 1e6:.4f} M")
print("Dynamic avg FLOPs per sample :", f"{test_dynamic['avg_flops'] / 1e6:.4f} M")

print("\n================ Route Ratio ================")
print("Early route ratio:", test_dynamic["early_route_ratio"])
print("Full route ratio :", test_dynamic["full_route_ratio"])

print("\nClassification Report: Dynamic Routing")
print(
    classification_report(
        test_dynamic["y_true"],
        test_dynamic["y_pred"],
        target_names=ACTIVITY_NAMES,
        digits=4,
    )
)

cm = confusion_matrix(test_dynamic["y_true"], test_dynamic["y_pred"])
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in ACTIVITY_NAMES],
    columns=[f"pred_{name}" for name in ACTIVITY_NAMES],
)

print("\nConfusion Matrix: Dynamic Routing")
try:
    display(cm_df)
except Exception:
    print(cm_df.to_string())


# ============================================================
# 15. Route analysis by activity class
# ============================================================

route_df = pd.DataFrame({
    "true_label": test_dynamic["y_true"],
    "pred_label": test_dynamic["y_pred"],
    "route": test_dynamic["route"],
    "gate_prob": test_dynamic["gate_prob"],
})

route_df["activity"] = route_df["true_label"].apply(lambda i: ACTIVITY_NAMES[i])
route_df["route_name"] = route_df["route"].apply(lambda r: "Full-Mamba" if r == 1 else "Early-Mamba")
route_df["correct"] = route_df["true_label"] == route_df["pred_label"]

route_summary = (
    route_df
    .groupby("activity")
    .agg(
        samples=("activity", "count"),
        full_route_ratio=("route", "mean"),
        avg_benefit_score=("gate_prob", "mean"),
        dynamic_accuracy=("correct", "mean"),
    )
    .reset_index()
    .sort_values("full_route_ratio", ascending=False)
)

print("\nRoute Summary by Activity")
try:
    display(route_summary)
except Exception:
    print(route_summary.to_string(index=False))


# ============================================================
# 16. Benefit-gate diagnostic table
# ============================================================

diagnostic_df = pd.DataFrame({
    "true_label": test_all["y_true"],
    "early_pred": test_all["early_pred"],
    "final_pred": test_all["final_pred"],
    "gate_prob": test_all["gate_prob"],
    "gate_target": test_all["gate_target"],
})

diagnostic_df["activity"] = diagnostic_df["true_label"].apply(lambda i: ACTIVITY_NAMES[i])
diagnostic_df["early_correct"] = diagnostic_df["early_pred"] == diagnostic_df["true_label"]
diagnostic_df["final_correct"] = diagnostic_df["final_pred"] == diagnostic_df["true_label"]
diagnostic_df["full_corrects_early"] = (~diagnostic_df["early_correct"]) & diagnostic_df["final_correct"]
diagnostic_df["full_harms_early"] = diagnostic_df["early_correct"] & (~diagnostic_df["final_correct"])

diagnostic_summary = (
    diagnostic_df
    .groupby("activity")
    .agg(
        samples=("activity", "count"),
        early_acc=("early_correct", "mean"),
        final_acc=("final_correct", "mean"),
        full_corrects_early_ratio=("full_corrects_early", "mean"),
        full_harms_early_ratio=("full_harms_early", "mean"),
        benefit_target_ratio=("gate_target", "mean"),
        avg_gate_prob=("gate_prob", "mean"),
    )
    .reset_index()
    .sort_values("benefit_target_ratio", ascending=False)
)

print("\nBenefit-Gate Diagnostic Summary by Activity")
try:
    display(diagnostic_summary)
except Exception:
    print(diagnostic_summary.to_string(index=False))


# ============================================================
# 17. Save model
# ============================================================

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

save_obj = {
    "model_state_dict": model.state_dict(),
    "config": {
        "model": "BenefitGatedSharedMambaHAR",
        "num_channels": NUM_CHANNELS,
        "seq_len": SEQ_LEN,
        "num_classes": NUM_CLASSES,
        "activity_names": ACTIVITY_NAMES,
        "hidden_dim": HIDDEN_DIM,
        "total_mamba_blocks": TOTAL_MAMBA_BLOCKS,
        "early_exit_blocks": EARLY_EXIT_BLOCKS,
        "dropout": DROPOUT,
        "early_loss_weight": EARLY_LOSS_WEIGHT,
        "gate_loss_weight": GATE_LOSS_WEIGHT,
        "compute_penalty_weight": COMPUTE_PENALTY_WEIGHT,
        "benefit_margin": BENEFIT_MARGIN,
        "benefit_tau": benefit_tau,
        "max_full_route_ratio": MAX_FULL_ROUTE_RATIO,
        "gate_hidden_dim": GATE_HIDDEN_DIM,
        "flops_info": flops_info,
    },
    "train_mean": train_mean,
    "train_std": train_std,
    "history": history,
    "tau_records": tau_df,
}

torch.save(save_obj, MODEL_SAVE_PATH)
print("\nSaved model to:", MODEL_SAVE_PATH)


# ============================================================
# 18. Single-window inference example
# ============================================================

@torch.no_grad()
def predict_one_window(model, x_np, benefit_tau):
    model.eval()

    x = torch.tensor(x_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    logits, route, gate_prob = model.forward_dynamic(
        x,
        benefit_tau=benefit_tau,
    )

    prob = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred_id = int(prob.argmax())

    route_id = int(route.item())
    route_name = "Full-Mamba" if route_id == 1 else "Early-Mamba"

    used_flops = (
        flops_info["full_route_dynamic"]
        if route_id == 1
        else flops_info["early_route"]
    )

    return {
        "pred_id": pred_id,
        "pred_activity": ACTIVITY_NAMES[pred_id],
        "benefit_score": float(gate_prob.item()),
        "route": route_name,
        "used_flops_M": used_flops / 1e6,
        "probability": prob,
    }


sample_idx = 0

example_result = predict_one_window(
    model,
    X_test[sample_idx],
    benefit_tau=benefit_tau,
)

print("\nSingle-window inference example")
print("True activity:", ACTIVITY_NAMES[y_test[sample_idx]])
print("Prediction:", example_result["pred_activity"])
print("Route:", example_result["route"])
print("Benefit score:", example_result["benefit_score"])
print("Used FLOPs:", f"{example_result['used_flops_M']:.4f} M")

Device: cuda
Full train X: (7352, 9, 128)
Test X      : (2947, 9, 128)
Activities  : ['WALK', 'UP', 'DOWN', 'SIT', 'STAND', 'LAY']
Train subjects for fitting: [1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 19, 21, 22, 23, 27, 28, 29]
Val subjects for gate calibration: [17, 25, 26, 30]
Train X: (5800, 9, 128)
Val X  : (1552, 9, 128)
Normalized train mean: 2.4871607820387e-05
Normalized train std : 1.0003135204315186
BenefitGatedSharedMambaHAR(
  (input_proj): Sequential(
    (0): Conv1d(9, 64, kernel_size=(1,), stride=(1,), bias=False)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU()
  )
  (blocks): ModuleList(
    (0-2): 3 x MiniMambaBlock(
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (in_proj): Linear(in_features=64, out_features=128, bias=True)
      (depthwise_conv): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), groups=64)
      (dt_proj): Linear(in_features=64, out_features=64, bias=True)
      (b

,epoch,train_loss,train_gate_loss,train_gate_target_ratio,train_gate_prob_mean,train_early_acc,train_early_macro_f1,train_final_acc,train_final_macro_f1,val_early_acc,val_early_macro_f1,val_final_acc,val_final_macro_f1,val_gate_target_ratio,val_gate_prob_mean,lr
70,71,0.068183,0.046402,0.157241,0.156333,0.961897,0.964114,0.993448,0.993979,0.929124,0.931942,0.950387,0.953407,0.138531,0.162765,6.180866e-05
71,72,0.062453,0.043137,0.155862,0.155099,0.964138,0.966492,0.994483,0.994930,0.926546,0.929420,0.951675,0.954607,0.130799,0.158470,4.894348e-05
72,73,0.066732,0.046193,0.153103,0.153512,0.962759,0.965139,0.992241,0.992871,0.929124,0.931854,0.949098,0.952213,0.130799,0.160813,3.754476e-05
73,74,0.064734,0.043726,0.157241,0.155096,0.961552,0.963825,0.995000,0.995403,0.928479,0.931313,0.951031,0.953999,0.129510,0.157383,2.763008e-05
74,75,0.060534,0.043359,0.150862,0.151203,0.964655,0.966813,0.995172,0.995563,0.925902,0.928787,0.951031,0.953980,0.134021,0.159742,1.921472e-05
75,76,0.058471,0.039940,0.153276,0.151369,0.965862,0.968194,0.995345,0.995722,0.929768,0.932629,0.950387,0.953457,0.132088,0.159195,1.231166e-05
76,77,0.060391,0.042737,0.154655,0.154198,0.967586,0.969711,0.995000,0.995406,0.932345,0.935077,0.950387,0.953328,0.137242,0.163145,6.931543e-06
77,78,0.063941,0.048231,0.155345,0.149958,0.962931,0.965138,0.994828,0.995244,0.925902,0.928926,0.950387,0.953252,0.130155,0.153522,3.082666e-06
78,79,0.061666,0.043183,0.151207,0.150960,0.964828,0.967016,0.993276,0.993821,0.929768,0.932650,0.949742,0.952859,0.131443,0.156811,7.709638e-07
79,80,0.058652,0.039563,0.153103,0.151669,0.963276,0.965565,0.995345,0.995721,0.929768,0.932571,0.949742,0.952815,0.132732,0.158609,0.000000e+00



Calibrated benefit-gate threshold
Tau: 0.7533088612556458
Val dynamic Acc: 0.9600515463917526
Val dynamic Macro-F1: 0.962401425366386
Val full-route ratio: 0.07474226804123711
Val average FLOPs: 7.9189 M

Top threshold candidates:


,tau,macro_f1,acc,full_route_ratio,avg_flops_M
186,0.753309,0.962401,0.960052,0.074742,7.918889
185,0.663423,0.962401,0.960052,0.079253,7.979636
184,0.609206,0.962401,0.960052,0.084407,8.049061
183,0.529760,0.962401,0.960052,0.089562,8.118485
182,0.459133,0.962401,0.960052,0.094716,8.187910
181,0.412315,0.962401,0.960052,0.099227,8.248657
180,0.323483,0.962401,0.960052,0.104381,8.318082
179,0.287440,0.962401,0.960052,0.109536,8.387507
187,0.786211,0.962307,0.960052,0.069588,7.849464
199,0.999253,0.962227,0.960052,0.010309,7.051079



================ Final Test Results ================
Early-only Acc      : 0.9385816084153377
Early-only Macro-F1 : 0.938598027625133
Full-only Acc       : 0.9355276552426196
Full-only Macro-F1  : 0.9352847343346579
Dynamic Acc         : 0.9368849677638276
Dynamic Macro-F1    : 0.9366801868175544

================ Benefit Gate ================
Test gate target ratio: 0.09229724854230881
Test gate prob mean   : 0.12203904241323471
Benefit threshold tau : 0.7533088612556458

================ Route FLOPs ================
Early route FLOPs per sample : 6.9122 M
Full route FLOPs per sample  : 20.3806 M
Full model once FLOPs        : 20.3251 M
Dynamic avg FLOPs per sample : 7.8537 M

================ Route Ratio ================
Early route ratio: 0.9300984051577876
Full route ratio : 0.06990159484221242

Classification Report: Dynamic Routing
              precision    recall  f1-score   support

        WALK     1.0000    0.9536    0.9763       496
          UP     0.9563    0.9766    0.9

...


Route Summary by Activity


,activity,samples,full_route_ratio,avg_benefit_score,dynamic_accuracy
4,UP,471,0.242038,0.331453,0.976645
0,DOWN,420,0.104762,0.143729,0.959524
5,WALK,496,0.064516,0.092842,0.953629
2,SIT,491,0.022403,0.112802,0.849287
3,STAND,532,0.009398,0.073155,0.885338
1,LAY,537,0.000000,0.005242,1.000000



Benefit-Gate Diagnostic Summary by Activity


,activity,samples,early_acc,final_acc,full_corrects_early_ratio,full_harms_early_ratio,benefit_target_ratio,avg_gate_prob
4,UP,471,0.942675,0.976645,0.033970,0.000000,0.297240,0.331453
2,SIT,491,0.841141,0.851324,0.010183,0.000000,0.124236,0.112802
0,DOWN,420,0.992857,0.957143,0.000000,0.035714,0.119048,0.143729
3,STAND,532,0.885338,0.883459,0.000000,0.001880,0.030075,0.073155
5,WALK,496,0.975806,0.947581,0.000000,0.028226,0.010081,0.092842
1,LAY,537,1.000000,1.000000,0.000000,0.000000,0.000000,0.005242



Saved model to: /content/drive/MyDrive/Colab Notebooks/HAR/BG-HAR/pt_files/uci_har.pt

Single-window inference example
True activity: STAND
Prediction: STAND
Route: Early-Mamba
Benefit score: 0.0027307304553687572
Used FLOPs: 6.9122 M
